## Module 1: Environment Setup and Dependency Injection
This module initializes the workspace for the **Automatic Drum Transcription (ADT)** project. It includes the installation of specialized libraries for audio source separation (**Demucs**) and hyperparameter optimization (**Keras Tuner**). Furthermore, it establishes the connection to the dataset repository on Google Drive and defines the directory structure for different drum components (Kick, Snare) from multiple sources (IDMT-SMT and Kaggle), ensuring a robust data pipeline for the subsequent training and analysis phases.

**To run the code add a shortcut to the Project folder in your own drive (My Drive folder)**

In [ ]:
!pip install -U demucs keras-tuner scikit-learn pretty_midi -q

# System and File Management
from google.colab import drive
import os
import shutil
import subprocess
import glob
from IPython.display import Audio, display
import time
import sys
from tqdm.notebook import tqdm

# Audio Analysis and Processing
import librosa
import librosa.display
import scipy.signal
from scipy.signal import butter, lfilter, filtfilt, sosfiltfilt
import pretty_midi

# Numerical Computation and Visualization
import numpy as np
import matplotlib.pyplot as plt
import random
import seaborn as sns

# Deep Learning Framework (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import mido

In [ ]:
# Mount Google Drive to access project datasets and save results
drive.mount('/content/drive')

# Define global paths for datasets and output directories
BASE_PROJECT_PATH = '/content/drive/MyDrive/Project/Task_1_2'
DATASET_PATH = f'{BASE_PROJECT_PATH}/Datasets'
RESULTS_DIR = f'{BASE_PROJECT_PATH}/Results'
MODEL_SAVE_DIR = f'{RESULTS_DIR}/Models'

# IDMT-SMT-DRUMS-V2: Professional dataset for ADT
# Kaggle: Supplementary data for model generalization
KAGGLE_KICK_PATH = f'{DATASET_PATH}/Kaggle/kick'
KAGGLE_SNARE_PATH = f'{DATASET_PATH}/Kaggle/snare'
IDMT_BASE_PATH = f'{DATASET_PATH}/IDMT-SMT-DRUMS-V2/audio'

# Create directories if they don't exist
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

## Module 2: Data Engineering and Advanced Augmentation Pipeline
This module implements a robust data processing pipeline designed to handle multi-source datasets (IDMT-SMT and Kaggle). It addresses the core challenges of **Automatic Drum Transcription (ADT)** by converting raw audio signals into time-frequency representations (**Mel Spectrograms**). To ensure model generalization and prevent overfitting, the pipeline incorporates both **Audio Domain Augmentation** (Time Stretching, Pitch Shifting) and **Vision Domain Augmentation** (SpecAugment), while strictly maintaining a separation between training and validation sets at the file-path level to prevent data leakage.

In [ ]:
TARGET_FRAMES = 15
SR = 44100

# DATA SCANNING
def get_audio_file_list(idmt_base_path, extra_kick_path, extra_snare_path):
    file_list = []
    drum_map_idmt = {'#KD': 0, '#SD': 1}

    if os.path.exists(idmt_base_path):
        for f in os.listdir(idmt_base_path):
            if f.endswith('.wav') and '#MIX' not in f:
                for suffix, label in drum_map_idmt.items():
                    if suffix in f:
                        file_list.append((os.path.join(idmt_base_path, f), label))
                        break

    for path, label in [(extra_kick_path, 0), (extra_snare_path, 1)]:
        if os.path.exists(path):
            for f in os.listdir(path):
                if f.endswith('.wav'):
                    file_list.append((os.path.join(path, f), label))
    return file_list

# FEATURE EXTRACTION ENGINE
def process_audio_list(file_list, augment=False, aug_factor=5):
    X, y = [], []
    desc = "Augmenting Data" if augment else "Processing Originals"

    for file_path, label in tqdm(file_list, desc=desc):
        try:
            audio_orig, _ = librosa.load(file_path, sr=SR, duration=0.30)
            if len(audio_orig) < SR * 0.05: continue

            iterations = aug_factor if augment else 1
            for _ in range(iterations):
                audio = audio_orig.copy()
                if augment:
                    if np.random.random() < 0.5:
                        audio = librosa.effects.time_stretch(audio, rate=np.random.uniform(0.8, 1.2))
                    if np.random.random() < 0.7:
                        audio = librosa.effects.pitch_shift(audio, sr=SR, n_steps=np.random.uniform(-2, 2))
                    audio += np.random.uniform(0.001, 0.005) * np.random.randn(len(audio))
                    audio *= np.random.uniform(0.7, 1.2)

                mel_spec = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=64, hop_length=512)
                mel_db = librosa.amplitude_to_db(mel_spec, ref=1.0)
                mel_norm = np.clip((mel_db + 80) / 80, 0, 1)

                if augment:
                    # SpecAugment: Frequency Mask
                    if np.random.random() < 0.5:
                        f_bin = np.random.randint(1, 8)
                        f_start = np.random.randint(0, 64 - f_bin)
                        mel_norm[f_start:f_start+f_bin, :] = 0
                    # SpecAugment: Time Mask
                    if np.random.random() < 0.5 and mel_norm.shape[1] > 2:
                        t_bin = np.random.randint(1, 3)
                        t_start = np.random.randint(0, mel_norm.shape[1] - t_bin)
                        mel_norm[:, t_start:t_start+t_bin] = 0

                if mel_norm.shape[1] >= TARGET_FRAMES:
                    mel_final = mel_norm[:, :TARGET_FRAMES]
                else:
                    mel_final = np.pad(mel_norm, ((0, 0), (0, TARGET_FRAMES - mel_norm.shape[1])))

                X.append(mel_final)
                y.append(label)
        except Exception: continue
    return np.array(X), np.array(y)

In [ ]:
all_files = get_audio_file_list(IDMT_BASE_PATH, KAGGLE_KICK_PATH, KAGGLE_SNARE_PATH)
train_paths, val_paths = train_test_split(all_files, test_size=0.2, random_state=42, stratify=[f[1] for f in all_files])

X_train, y_train = process_audio_list(train_paths, augment=True, aug_factor=6)
X_val, y_val = process_audio_list(val_paths, augment=False)

def print_dataset_summary(y_train, y_val):
    train_kicks = np.count_nonzero(y_train == 0)
    train_snares = np.count_nonzero(y_train == 1)

    val_kicks = np.count_nonzero(y_val == 0)
    val_snares = np.count_nonzero(y_val == 1)

    print("SAMPLES:")
    print("-" * 40)
    print(f" TRAINING SET ({len(y_train)} samples):")
    print(f"   -  Kicks:  {train_kicks}")
    print(f"   -  Snares: {train_snares}")

    print("-" * 40)
    print(f" VALIDATION SET ({len(y_val)} samples):")
    print(f"   -  Kicks:  {val_kicks}")
    print(f"   -  Snares: {val_snares}")
    print("-" * 40)


print_dataset_summary(y_train, y_val)

def plot_sample_features(X, y, samples_to_show=2):
    plt.figure(figsize=(12, 5))
    for i, label_name in enumerate(['Kick (Label 0)', 'Snare (Label 1)']):
        idx = np.where(y == i)[0][0] # Get first occurrence
        plt.subplot(1, 2, i+1)
        librosa.display.specshow(X[idx], sr=SR, hop_length=512, x_axis='time', y_axis='mel')
        plt.title(f"Normalized Mel Spectrogram: {label_name}")
        plt.colorbar(format='%+2.0f')
    plt.tight_layout()
    plt.show()

print("\nVisualizing samples from the augmented training set:")
plot_sample_features(X_train, y_train)

# Final Reshape
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]

## Module 3: Adaptive CNN Architecture and Hyperparameter Optimization
This module defines a dynamic **Convolutional Neural Network (CNN)** architecture optimized through **Keras Tuner (Hyperband)**. The model is specifically designed for binary classification of drum components (Kick vs. Snare). Key architectural features include **Batch Normalization** for training stability, **Dropout** for regularization, and a flexible number of convolutional blocks. The training pipeline employs an automated search for the best learning rate and filter configuration, followed by a final training stage with **ReduceLROnPlateau** and **EarlyStopping** to ensure the highest possible generalization on the validation set.

In [ ]:
# ARCHITECTURE DEFINITION
def model_builder(hp):
    """
    Builds a dynamic CNN architecture for hyperparameter tuning.
    Inputs: hp (HyperParameters object)
    """
    model = models.Sequential()

    # Dynamic Input Shape based on pre-processed Mel Spectrograms
    model.add(layers.Input(shape=X_train.shape[1:]))

    # Convolutional Block 1
    hp_filters_1 = hp.Int('filters_1', min_value=16, max_value=32, step=16)
    model.add(layers.Conv2D(hp_filters_1, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Convolutional Block 2
    hp_filters_2 = hp.Int('filters_2', min_value=32, max_value=64, step=32)
    model.add(layers.Conv2D(hp_filters_2, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Optional Convolutional Block 3
    if hp.Boolean('add_3rd_block'):
        hp_filters_3 = hp.Int('filters_3', min_value=64, max_value=128, step=32)
        model.add(layers.Conv2D(hp_filters_3, (3, 3), activation='relu', padding='same'))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Flatten())

    # Fully Connected Layers
    hp_units = hp.Int('units', min_value=32, max_value=128, step=32)
    model.add(layers.Dense(units=hp_units, activation='relu'))

    # High Dropout rate to combat overfitting on augmented audio data
    hp_dropout = hp.Float('dropout', min_value=0.3, max_value=0.6, step=0.1)
    model.add(layers.Dropout(hp_dropout))

    # Binary Output Layer (0 = Kick, 1 = Snare)
    model.add(layers.Dense(1, activation='sigmoid'))

    # Learning Rate Optimization
    hp_lr = hp.Choice('learning_rate', values=[1e-3, 5e-4, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
model_filename = 'drum_cnn_v1.keras'
full_model_path = os.path.join(MODEL_SAVE_DIR, model_filename)

if os.path.exists(full_model_path):
    print(f"Existing model found at: {full_model_path}")
    print("Loading pre-trained model to skip training...")
    final_model = tf.keras.models.load_model(full_model_path)

    RUN_TRAINING = False
else:
    print("No pre-trained model found. Starting Hyperparameter Search and Training...")
    RUN_TRAINING = True


if RUN_TRAINING:

    # TUNER CONFIGURATION (Hyperband)
    tuner = kt.Hyperband(
        model_builder,
        objective='val_loss',
        max_epochs=30,
        factor=3,
        directory='tuning_dir',
        project_name='adt_optimization_v1',
        overwrite=True
    )


    # HYPERPARAMETER SEARCH

    print("\nSearching for the optimal architecture...")
    stop_early = callbacks.EarlyStopping(monitor='val_loss', patience=5)

    tuner.search(
        X_train, y_train,
        epochs=30,
        validation_data=(X_val, y_val),
        batch_size=32,
        callbacks=[stop_early]
    )

In [ ]:
#FINAL TRAINING OF THE BEST MODEL
if RUN_TRAINING:

    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
    final_model = tuner.hypermodel.build(best_hps)

    print(f"\nBest configuration found: LR={best_hps.get('learning_rate')}, Dropout={best_hps.get('dropout')}")

    # Callbacks for final training stage
    reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)

    print("\nStarting final model training...")
    history = final_model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_val, y_val),
        shuffle=True,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    model_filename = 'drum_cnn_v1.keras'
    full_model_path = os.path.join(MODEL_SAVE_DIR, model_filename)

    final_model.save(full_model_path)
    print(f"Model successfully saved to: {full_model_path}")

    #TRAINING EVALUATION PLOTS
    plt.figure(figsize=(14, 5))

    # Accuracy Plot
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linestyle='--')
    plt.title('Accuracy History')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1)
    plt.legend()

    # Loss Plot
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss', linestyle='--')
    plt.title('Loss History (Binary Crossentropy)')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.ylim(0, 1)
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
final_model.summary()

## Module 4: Audio Signal Analysis and Time-Frequency Visualization
This module initiates the **Automatic Drum Transcription (ADT)** pipeline by loading the target polyphonic audio track at a high-fidelity sampling rate of 44100 Hz. It performs a dual-domain exploratory analysis: **Temporal**, through waveform inspection to identify macro-dynamics, and **Spectral**, using a Short-Time Fourier Transform (**STFT**) to generate a logarithmic spectrogram. This preliminary step is essential for assessing the signal's frequency distribution and energy density before proceeding with source separation and component classification.

### **Choose here your song:**

In [ ]:
# Change SONG_NAME to process different audio files without overlapping results

#SONG_NAME = 'Billie_Jean'
#SONG_NAME = 'Yellow'
SONG_NAME = 'Mattia'

AUDIO_SOURCE_FILE = f'{DATASET_PATH}/{SONG_NAME}.mp3'
MIDI_PATH = f'{DATASET_PATH}/{SONG_NAME}.mid'

# Demucs Output Path (Separates tracks per song name)
SEPARATED_TRACKS_DIR = f'{DATASET_PATH}/htdemucs/{SONG_NAME}'

In [ ]:
#Audio Loading
audio_signal, sampling_rate = librosa.load(AUDIO_SOURCE_FILE, sr=44100)

print(f"{SONG_NAME.replace("_", " ")} original signal analysis:")
print(f"Sampling Rate (fs): {sampling_rate} Hz")
print(f"Duration: {librosa.get_duration(y=audio_signal, sr=sampling_rate):.2f} seconds")
print(f"Total samples: {len(audio_signal)}")

print("Playing original track:")
display(Audio(AUDIO_SOURCE_FILE))

# Global Visualization: Time vs Frequency Domains
plt.figure(figsize=(15, 12))

# Waveform
plt.subplot(2, 1, 1)
librosa.display.waveshow(audio_signal, sr=sampling_rate, color='royalblue', alpha=0.6)
plt.title(f'Waveform: {SONG_NAME.replace("_", " ")}')
plt.ylabel('Amplitude')
plt.xlabel('Time')
plt.grid(True, alpha=0.3)

# Spectrogram
plt.subplot(2, 1, 2)
# STFT and convert to dB
stft_magnitude = np.abs(librosa.stft(audio_signal))
spectrogram_db = librosa.amplitude_to_db(stft_magnitude, ref=np.max)

# logarithmic frequency axis
librosa.display.specshow(spectrogram_db, sr=sampling_rate, x_axis='time', y_axis='log', cmap='coolwarm')
plt.title(f'Log-Frequency Spectrogram: {SONG_NAME.replace("_", " ")}')
plt.colorbar(format='%+2.0f dB')

plt.tight_layout()
plt.show()

In [ ]:
# ANALYTICAL ZOOM AND SPECTRAL DETAIL

# Temporal Slicing
# Focused analysis on the first 10 seconds to inspect percussive transients
start_sec = 0.0
end_sec = 10.0
start_sample = int(start_sec * sampling_rate)
end_sample = int(end_sec * sampling_rate)

audio_zoom = audio_signal[start_sample:end_sample]

plt.figure(figsize=(15, 10))

# Zoomed Waveform
plt.subplot(2, 1, 1)
librosa.display.waveshow(audio_zoom, sr=sampling_rate, color='royalblue', alpha=0.8)
plt.title(f'ZOOMED WAVEFORM: Initial {end_sec}s')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.4)

# Logarithmic Spectrogram (Kick vs Snare Timbre Detail)
plt.subplot(2, 1, 2)

# Setting n_fft to 2048 provides adequate frequency binning for low-end analysis
stft_zoom = np.abs(librosa.stft(audio_zoom, n_fft=2048, hop_length=512))
db_zoom = librosa.amplitude_to_db(stft_zoom, ref=np.max)

# Visualization with Logarithmic Y-axis to separate low-frequency components
librosa.display.specshow(db_zoom, sr=sampling_rate, x_axis='time', y_axis='log', cmap='magma')
plt.title(f'ZOOMED SPECTROGRAM: Spectral distribution (Kick vs. Snare)')
plt.colorbar(format='%+2.0f dB')

# Note: The log scale facilitates the distinction between Kick (approx 60Hz) and Snare (approx 200Hz+)
plt.tight_layout()
plt.show()

## Module 5: Audio Source Separation via Hybrid Transformer Demucs
This module implements the **Source Separation** stage using the state-of-the-art **Demucs (htdemucs)** model. The objective is to isolate the drum stem from the original polyphonic mix to minimize interference from other instruments (e.g., bass, vocals) during transcription.

In [ ]:
# SOURCE SEPARATION AND DRUM TRACK ISOLATION

# PATH CONFIGURATION
final_file_path = os.path.join(SEPARATED_TRACKS_DIR, "drums.mp3")
os.makedirs(SEPARATED_TRACKS_DIR, exist_ok=True)

# DEMUCS EXECUTION LOGIC
if os.path.exists(final_file_path):
    print(f"File already exists: {final_file_path}")
    print("Skipping separation process to save computation time.")
else:
    print(f"Initiating source separation. Target directory: {SEPARATED_TRACKS_DIR}")

    # Temporary local storage for processing
    temp_out = "/content/temp_demucs"

    # htdemucs command (High-quality Hybrid Transformer model)
    command = f'python -m demucs.separate -n htdemucs --mp3 "{AUDIO_SOURCE_FILE}" -o {temp_out}'

    try:
        print("Demucs processing in progress (this may take several minutes)...")
        process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

        # Monitor process output
        while True:
            line = process.stdout.readline()
            if not line and process.poll() is not None:
                break
            if line:
                # Log only essential status updates
                if "Separating" in line or "Writing" in line or "Selected model" in line:
                    print(f"   > {line.strip()}")

        process.wait()

        # Relocate generated file to persistent storage
        generated_drums = glob.glob(f"{temp_out}/htdemucs/*/drums.mp3")[0]
        shutil.move(generated_drums, final_file_path)

        # Cleanup temporary directory
        if os.path.exists(temp_out):
            shutil.rmtree(temp_out)

        print("Separation completed successfully.")

    except Exception as e:
        print(f"Error during Demucs execution: {e}")

In [ ]:
# ISOLATED SIGNAL ANALYSIS

if os.path.exists(final_file_path):
    # Load and Normalize the isolated drum track
    y_drums, sr_drums = librosa.load(final_file_path, sr=44100)
    y_drums = librosa.util.normalize(y_drums)

    print(f"Status: Drum track loaded. Samples: {len(y_drums)} | SR: {sr_drums}")
    display(Audio(y_drums, rate=sr_drums))

    plt.figure(figsize=(15, 10))

    # Waveform of the isolated drum stem
    plt.subplot(2, 1, 1)
    librosa.display.waveshow(y_drums, sr=sr_drums, color='orange', alpha=0.7)
    plt.title('Waveform: Isolated Drum Track')
    plt.ylabel('Normalized Amplitude')
    plt.grid(True, alpha=0.3)

    # Log-Frequency Spectrogram of the isolated drum stem
    plt.subplot(2, 1, 2)
    stft_drums = np.abs(librosa.stft(y_drums))
    db_drums = librosa.amplitude_to_db(stft_drums, ref=np.max)

    # Using log-scale to clearly visualize Kick (low freq) and Snare energy
    librosa.display.specshow(db_drums, sr=sr_drums, x_axis='time', y_axis='log', cmap='magma')
    plt.title('Log-Frequency Spectrogram: Isolated Drums')
    plt.colorbar(format='%+2.0f dB')

    plt.tight_layout()
    plt.show()

## Module 6: Deterministic Filtering and Onset Detection Pipeline
This module implements the core **Digital Signal Processing (DSP)** logic for event identification. A stable **Butterworth Low-Pass Filter** is applied to the isolated drum track with a strategic cutoff frequency of 700 Hz; this threshold is chosen to preserve the harmonic body of the Snare while suppressing high-frequency interference from cymbals (Hi-Hats). Subsequently, an **Onset Detection** algorithm with adaptive thresholding and backtracking is executed to locate the precise temporal coordinates of each drum hit, providing the foundation for the automated transcription process.

In [ ]:
# STABLE FILTERING AND TEMPORAL ONSET DETECTION

# STABLE FILTER DEFINITION
def butter_lowpass_stable(data, cutoff, fs, order=4):
    """
    Implements a zero phase low-pass filter using SOS for numerical stability.
    Designed to suppress Hi-Hat frequencies while preserving Kick and Snare.
    """
    nyquist = 0.5 * fs
    normalized_cutoff = cutoff / nyquist
    # Use Second-Order Sections for superior stability in digital filters
    sos = butter(order, normalized_cutoff, btype='low', analog=False, output='sos')
    return sosfiltfilt(sos, data)

# GLOBAL SIGNAL PROCESSING
print("Initiating DSP analysis on the full track...")

# 700Hz Filtering
# Motivation: Snare drums possess critical harmonic components between 200Hz and 600Hz.
# A higher cutoff ensures the CNN receives sufficient spectral information to distinguish Snare from Kick.
y_clean_full = butter_lowpass_stable(y_drums, cutoff=700, fs=sr_drums)

# Global Onset Detection
# wait=10: Prevents re-triggering within 100ms
# delta=0.3: Adaptive threshold to ignore residual background noise
# backtrack=True: Shifts detection to the exact onset start, ensuring transient integrity
print("Detecting transients with active backtracking...")
onset_frames = librosa.onset.onset_detect(
    y=y_clean_full,
    sr=sr_drums,
    delta=0.28,
    wait=6,
    backtrack=True
)
onset_times = librosa.frames_to_time(onset_frames, sr=sr_drums)

#TRANSCRIPTION STATISTICS
total_duration = librosa.get_duration(y=y_clean_full, sr=sr_drums)
num_hits = len(onset_times)
rhythmic_density = (num_hits / total_duration) * 60

print("\nANALYSIS SUMMARY:")
print(f"   - Track Duration: {total_duration:.2f} s")
print(f"   - Detected Onsets: {num_hits}")
print(f"   - Average Rhythmic Density: {rhythmic_density:.1f} events/min")

# HYBRID VISUALIZATION
plt.figure(figsize=(15, 10))

# Rhythmic Overview
plt.subplot(2, 1, 1)
onset_env = librosa.onset.onset_strength(y=y_clean_full, sr=sr_drums)
times_env = librosa.times_like(onset_env, sr=sr_drums)
plt.plot(times_env, onset_env, color='blue', alpha=0.6, label='Energy Flux')
plt.title(f'Global Rhythmic Overview ({num_hits} events detected)')
plt.xlabel('Time (s)')
plt.ylabel('Energy Intensity')
plt.grid(True, alpha=0.3)

# Precision Zoom (0s - 10s)
# Used for manual verification of detection accuracy
zoom_start, zoom_end = 0, 10
mask = (times_env >= zoom_start) & (times_env <= zoom_end)
onsets_zoom = onset_times[(onset_times >= zoom_start) & (onset_times <= zoom_end)]

plt.subplot(2, 1, 2)
# Plot filtered signal in the zoomed window
zoom_samples_start = int(zoom_start * sr_drums)
zoom_samples_end = int(zoom_end * sr_drums)
librosa.display.waveshow(y_clean_full[zoom_samples_start:zoom_samples_end],
                         sr=sr_drums, offset=zoom_start, color='orange', alpha=0.7, label='Filtered Signal (<700Hz)')
plt.vlines(onsets_zoom, -1, 1, color='red', linestyle='--', linewidth=2, label='Detected Onsets')
plt.title(f'Precision Detail (Zoom {zoom_start}-{zoom_end}s) - Ground Truth Verification')
plt.xlabel('Time (s)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Module 7: Feature Extraction for Inference and Tensor Shaping
This module implements the **Sliding Window Extraction** logic to prepare the real-world audio data for the CNN. For each temporal coordinate identified in the onset detection phase, a fixed-duration segment (approximately 175ms) is extracted from the isolated drum track. These segments are converted into **Mel Spectrograms** using parameters identical to those used during the training phase (N_mels=64, TARGET\_FRAMES=15). The module ensures that every input patch is strictly shaped as a (64, 15, 1) tensor, applying zero-padding where necessary to maintain consistency within the deep learning inference pipeline.

In [ ]:
# PATCH EXTRACTION AND CNN FEATURE GENERATION

def extract_inference_features(audio_data, onset_frames, sampling_rate):
    """
    Extracts spectral patches from detected onset timestamps.
    Uses the full-bandwidth drum track to preserve snare harmonics.
    """
    features = []
    hop_length = 512
    TARGET_FRAMES = 15 # Must match training dimensions strictly

    print(f"Feature Extraction: Processing {len(onset_frames)} onsets...")

    for frame in onset_frames:
        # Convert onset frame index to audio sample index
        start_sample = librosa.frames_to_samples(frame, hop_length=hop_length)

        # Calculate sample duration required to achieve exactly TARGET_FRAMES
        target_duration_samples = hop_length * (TARGET_FRAMES - 1)
        end_sample = start_sample + target_duration_samples

        # Audio segment extraction with tail padding for end-of-file hits
        if end_sample > len(audio_data):
            segment = audio_data[start_sample:]
            padding = target_duration_samples - len(segment)
            segment = np.pad(segment, (0, padding))
        else:
            segment = audio_data[start_sample:end_sample]

        # SPECTROGRAM COMPUTATION
        # Parameters must be identical to training configuration
        mel_spec = librosa.feature.melspectrogram(
            y=segment,
            sr=sampling_rate,
            n_mels=64,
            n_fft=2048,
            hop_length=hop_length
        )

        # Logarithmic normalization to [0, 1] range based on -80dB floor
        mel_db = librosa.amplitude_to_db(mel_spec, ref=1.0)
        mel_norm = (mel_db + 80) / 80
        mel_norm = np.clip(mel_norm, 0, 1)

        #TENSOR SHAPING
        # Compensate for STFT rounding variations to ensure fixed shape (64, 15)
        if mel_norm.shape[1] >= TARGET_FRAMES:
            features.append(mel_norm[:, :TARGET_FRAMES])
        else:
            pad_width = TARGET_FRAMES - mel_norm.shape[1]
            features.append(np.pad(mel_norm, ((0, 0), (0, pad_width))))

    return np.array(features)


print("Generating patches for CNN inference...")
X_inference = extract_inference_features(y_drums, onset_frames, sr_drums)

# Add channel dimension for Keras compatibility: (Batch, Height, Width, Channels)
X_inference_cnn = X_inference[..., np.newaxis]

print(f"Tensor generation complete. Input shape: {X_inference_cnn.shape}")

# Visualization of the first few patches to verify feature quality
plt.figure(figsize=(15, 3))
for i in range(min(5, len(X_inference))):
    plt.subplot(1, 5, i+1)
    librosa.display.specshow(X_inference[i], sr=sr_drums, hop_length=512, cmap='magma')
    plt.title(f"Patch {i}")
    plt.axis('off')
plt.suptitle("CNN Input Patches (Mel Spectrograms)")
plt.show()

## Module 8: Hybrid Transcription Engine and Physical Validation
This module implements the final **Inference and Post-Processing** pipeline. It adopts a "Model-First" logic, where the CNN classifies each detected onset, followed by a **Dynamic Thresholding** mechanism to adapt to the specific gain of the track. A unique **Physical Validation** step is introduced: the system calculates the **Low-Energy Ratio** for each hit, comparing the energy in the low-frequency spectrum (< 600 Hz) against the full bandwidth. This hybrid approach allows for the detection of "Physical Doubts," where the AI's prediction contradicts the acoustic properties of the signal, ensuring a highly robust final transcription.

In [ ]:
# HYBRID TRANSCRIPTION AND PERSISTENT EXPORT

def transcribe_full_track(X_features, timestamps, model, audio_signal, sr):
    print("Initiating full track transcription...")

    # Safety check for CNN input dimensions
    if X_features.ndim == 3:
        X_features = X_features[..., np.newaxis]
        print(f"   -> Input reshaped to {X_features.shape} for CNN compatibility.")

    #Neural Network Inference
    raw_predictions = model.predict(X_features, verbose=0).flatten()

    # Local Min-Max Normalization
    p_min, p_max = raw_predictions.min(), raw_predictions.max()
    denominator = (p_max - p_min) if (p_max - p_min) > 1e-9 else 1.0
    normalized_preds = (raw_predictions - p_min) / denominator

    # Compute Dynamic Median Threshold
    threshold = np.clip(np.median(normalized_preds), 0.2, 0.8)
    print(f"Calculated Dynamic Threshold: {threshold:.4f}")

    # Heuristic Filter (Low-Pass 600Hz) for Physical Validation
    sos = scipy.signal.butter(4, 600, 'low', fs=sr, output='sos')
    audio_lowpass = scipy.signal.sosfiltfilt(sos, audio_signal)

    kick_times, snare_times = [], []
    stats = {"kick": 0, "snare": 0, "phys_doubt": 0, "low_conf": 0}

    print(f"\n--- ANALYSIS EXCERPT (First 15 events) ---")
    print(f"{'Time':<8} | {'Prob':<6} | {'Classification':<15} | {'Low-Energy Ratio'}")
    print("-" * 75)

    # Processing each detected event
    for i, p in enumerate(normalized_preds):
        is_snare_cnn = p > threshold
        confidence = abs(p - threshold)
        if confidence < 0.1: stats["low_conf"] += 1

        # Spectral Energy Analysis (150ms window)
        start_idx = int(timestamps[i] * sr)
        end_idx = min(start_idx + int(sr * 0.15), len(audio_signal))

        seg_full = audio_signal[start_idx:end_idx]
        seg_low = audio_lowpass[start_idx:end_idx]

        # Calculate Energy Ratio
        energy_full = np.sum(seg_full**2) + 1e-9
        energy_low = np.sum(seg_low**2)
        low_energy_ratio = energy_low / energy_full

        is_phys_doubt = False
        if is_snare_cnn:
            label = "SNARE"
            snare_times.append(timestamps[i])
            stats["snare"] += 1
            if low_energy_ratio > 0.85: is_phys_doubt = True
        else:
            label = "KICK"
            kick_times.append(timestamps[i])
            stats["kick"] += 1
            if low_energy_ratio < 0.30: is_phys_doubt = True

        if is_phys_doubt: stats["phys_doubt"] += 1

        # Logging for verification
        if i < 15:
            status = "Doubt" if is_phys_doubt else ("Uncertain" if confidence < 0.1 else "OK")
            print(f"{timestamps[i]:>6.2f}s | {p:.2f}   | {label:<15} | {low_energy_ratio:.2f} ({status})")

    # Final Summary
    total_duration = len(audio_signal) / sr
    print("-" * 75)
    print(f"TRANSCRIPTION COMPLETED ({total_duration:.2f} sec)")
    print(f"Total Kicks: {stats['kick']} | Total Snares: {stats['snare']}")
    print(f"Physical Discrepancies: {stats['phys_doubt']}")

    # Visualization (Global and first 30s)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10))

    librosa.display.waveshow(audio_signal, sr=sr, ax=ax1, alpha=0.4, color='gray')
    ax1.vlines(kick_times, 0.2, 0.9, color='red', alpha=0.6, label='Kick (CNN)')
    ax1.vlines(snare_times, -0.9, -0.2, color='blue', alpha=0.6, label='Snare (CNN)')
    ax1.set_title("Global Transcription Overview")
    ax1.legend()

    librosa.display.waveshow(audio_signal, sr=sr, ax=ax2, alpha=0.5, color='gray')
    ax2.vlines(kick_times, 0.2, 0.9, color='red', alpha=0.8, linewidth=2)
    ax2.vlines(snare_times, -0.9, -0.2, color='blue', alpha=0.8, linewidth=2)
    ax2.set_xlim(0, 30)
    ax2.set_title("Detailed Transcription (First 30 Seconds)")

    plt.tight_layout()
    plt.show()

    return kick_times, snare_times


# Define the results directory on Google Drive
os.makedirs(RESULTS_DIR, exist_ok=True)

# Run Transcription
limit = min(len(X_inference_cnn), len(onset_times))
final_kicks, final_snares = transcribe_full_track(
    X_inference_cnn[:limit],
    onset_times[:limit],
    final_model,
    y_drums,
    sr_drums
)

# Export results to TXT on Drive
output_file_path = os.path.join(RESULTS_DIR, f"{SONG_NAME}_Transcription.txt")

try:
    with open(output_file_path, "w") as f:
        f.write("Time(s)\tInstrument\n")
        events = sorted([(t, "KICK") for t in final_kicks] + [(t, "SNARE") for t in final_snares])
        for t, label in events:
            f.write(f"{t:.3f}\t{label}\n")
    print(f"\nTranscription file successfully saved to Drive: {output_file_path}")
except Exception as e:
    print(f"Error saving transcription file: {e}")

## Module 9: MIR Metric Validation and Symbolic Alignment
This module performs an objective evaluation of the transcription system by comparing the CNN predictions against the **Ground Truth MIDI annotations**. It implements a **Symbolic Validation** approach, aligning detected onsets with their corresponding MIDI notes within a specific time window (0-30s). The module computes standard **Music Information Retrieval (MIR)** metrics, including **Precision, Recall, and F1-Score**, and generates a **Confusion Matrix** to identify specific classification biases between Kick and Snare components.

In [ ]:
import pretty_midi

def load_midi_ground_truth(midi_path):
    """
    Extracts Kick and Snare timestamps from a MIDI file for MIR validation.
    Maps GM pitches 35/36 (Kick) and 38/40 (Snare).
    """
    print(f"Loading MIDI Ground Truth: {midi_path}...")
    try:
        pm = pretty_midi.PrettyMIDI(midi_path)
    except Exception as e:
        print(f"Error loading MIDI file: {e}")
        return None, None

    # Identify the drum track via General MIDI flags or naming conventions
    drum_track = None
    for instrument in pm.instruments:
        if instrument.is_drum or "drum" in instrument.name.lower():
            drum_track = instrument
            print(f"Drum track identified: {instrument.name}")
            break

    if drum_track is None:
        print("Warning: No drum track detected.")
        return None, None

    gt_kick = []
    gt_snare = []

    # Standard General MIDI Mapping for ADT tasks
    for note in drum_track.notes:
        if note.pitch in [35, 36]:
            gt_kick.append(note.start)
        elif note.pitch in [38, 40]:
            gt_snare.append(note.start)

    return np.array(sorted(gt_kick)), np.array(sorted(gt_snare))

if SONG_NAME!= 'Mattia':
  # Output assigned to global variables used in validation modules
  gt_kicks_all, gt_snares_all = load_midi_ground_truth(MIDI_PATH)

  print("--- Ground Truth Extraction Summary ---")
  if gt_kicks_all is not None:
      print(f"Kick events extracted: {len(gt_kicks_all)}")
      print(f"Snare events extracted: {len(gt_snares_all)}")
  else:
      print("Error: Extraction failed.")

In [ ]:
# MIDI GROUND TRUTH VISUALIZATION (ZOOM 30s)
if SONG_NAME != 'Mattia':

    def plot_midi_events(kicks, snares, audio_signal=None, sr=None):
        """
        Visualizes Kick and Snare events extracted from the MIDI file.
        Focuses on the first 30 seconds for detailed alignment inspection.
        """
        plt.figure(figsize=(18, 8))

        # Global Event Strip
        plt.subplot(2, 1, 1)

        # Plot Kicks (Red)
        plt.scatter(kicks, np.zeros_like(kicks), color='red', marker='|', s=500, label='Kick (MIDI)', alpha=0.8)
        # Plot Snares (Blue)
        plt.scatter(snares, np.ones_like(snares), color='blue', marker='|', s=500, label='Snare (MIDI)', alpha=0.8)

        plt.yticks([0, 1], ['KICK', 'SNARE'])
        plt.ylim(-0.5, 1.5)
        plt.title(f"MIDI Ground Truth Events", fontsize=14)
        plt.xlabel("Time (s)")
        plt.grid(True, axis='x', alpha=0.3)
        plt.legend(loc='upper right')

        # Alignment Check
        plt.subplot(2, 1, 2)

        ZOOM_LIMIT = 30.0

        # Plot Audio Waveform (Background) if available
        if audio_signal is not None and sr is not None:
            librosa.display.waveshow(audio_signal, sr=sr, alpha=0.5, color='gray', label='Audio Waveform')

        # Filter events for the zoom window
        kicks_zoom = kicks[kicks < ZOOM_LIMIT]
        snares_zoom = snares[snares < ZOOM_LIMIT]

        # Overlay Vertical Lines
        plt.vlines(kicks_zoom, 0.2, 0.9, color='red', linewidth=2, linestyle='-', label='Kick (MIDI)')
        plt.vlines(snares_zoom, -0.9, -0.2, color='blue', linewidth=2, linestyle='-', label='Snare (MIDI)')

        plt.xlim(0, ZOOM_LIMIT)
        plt.title(f"Alignment Detail (First {ZOOM_LIMIT}s) - Waveform vs MIDI", fontsize=14)
        plt.xlabel("Time (s)")
        plt.legend(loc='upper right')
        plt.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    # Only run the plot if variables exist
    if 'gt_kicks_all' in locals() or 'gt_kicks_all' in globals():
        print(f"Visualizing MIDI events (First 30s) for: {os.path.basename(MIDI_PATH)}")

        # Use separated drums if available, else full mix
        ref_audio = y_drums if 'y_drums' in locals() or 'y_drums' in globals() else (audio_signal if 'audio_signal' in locals() or 'audio_signal' in globals() else None)
        ref_sr = SR if 'SR' in locals() or 'SR' in globals() else 44100

        plot_midi_events(gt_kicks_all, gt_snares_all, ref_audio, ref_sr)
    else:
        print("Error: Ground Truth variables not found. Please run the MIDI loading chunk first.")
else:
    print(f"Skipping visualization for {SONG_NAME} as requested.")

In [ ]:
# different acquisition of the midi file
if SONG_NAME == 'Mattia':
    try:
        import mido
    except ImportError:
        !pip install mido -q
        import mido

    def load_and_plot_robust(midi_path):
            mid = mido.MidiFile(midi_path)
            kicks = []
            snares = []
            current_time = 0.0

            # Mido sums delta times to calculate absolute timestamps
            for msg in mid:
                current_time += msg.time
                if msg.type == 'note_on' and msg.velocity > 0:
                    if msg.note == 36:   # Kick
                        kicks.append(current_time)
                    elif msg.note == 38: # Snare
                        snares.append(current_time)

            # Convert to arrays
            gt_kick = np.array(kicks)
            gt_snare = np.array(snares)

            # --- INTEGRATION: SHIFT TIMESTAMPS BASED ON TRIM ---
            if 'start_trim_sec' in globals():
                print(f"Applying temporal shift: -{start_trim_sec:.3f}s (Audio Trim Compensation)")
                gt_kick = gt_kick - start_trim_sec
                gt_snare = gt_snare - start_trim_sec
                # Keep only values > 0
                gt_kick = gt_kick[gt_kick >= 0]
                gt_snare = gt_snare[gt_snare >= 0]

            print(f"Data extracted for {SONG_NAME}:")
            print(f"   -> Kick (36): {len(gt_kick)}")
            print(f"   -> Snare (38): {len(gt_snare)}")

            # Update global variables for evaluation
            global gt_kicks_all, gt_snares_all
            gt_kicks_all = gt_kick
            gt_snares_all = gt_snare

            # VISUALIZATION
            plt.figure(figsize=(18, 8))
            my_sr = SR if 'SR' in globals() else 44100

            # Global View
            plt.subplot(2, 1, 1)
            plt.scatter(gt_kick, np.zeros_like(gt_kick), color='red', marker='|', s=500, label='Kick (36)', alpha=0.8)
            plt.scatter(gt_snare, np.ones_like(gt_snare), color='blue', marker='|', s=500, label='Snare (38)', alpha=0.8)
            plt.yticks([0, 1], ['KICK', 'SNARE'])
            plt.title(f"Ground Truth Events (Trimmed) - {len(gt_kick)+len(gt_snare)} Total")
            plt.legend()
            plt.grid(True, axis='x', alpha=0.3)

            # Zoom View (First 30s)
            plt.subplot(2, 1, 2)
            ZOOM = 30.0

            if 'y_drums' in globals():
                librosa.display.waveshow(y_drums, sr=my_sr, alpha=0.5, color='gray', label='Audio Drums')
            elif 'audio_signal' in globals():
                librosa.display.waveshow(audio_signal, sr=my_sr, alpha=0.5, color='gray', label='Audio Full')

            k_zoom = gt_kick[gt_kick < ZOOM]
            s_zoom = gt_snare[gt_snare < ZOOM]

            plt.vlines(k_zoom, 0.2, 0.9, color='red', linewidth=2, label='Kick')
            plt.vlines(s_zoom, -0.9, -0.2, color='blue', linewidth=2, label='Snare')
            plt.xlim(0, ZOOM)
            plt.title(f"Alignment Detail (First {ZOOM}s)")
            plt.legend()

            plt.tight_layout()
            plt.show()

            return gt_kick, gt_snare

    load_and_plot_robust(MIDI_PATH)

In [ ]:
# SCIENTIFIC VALIDATION AND PERFORMANCE METRICS

def validate_sequential_alignment(times, model, X_features, gt_kick, gt_snare, limit_sec=30.0):
    """
    Performs a step-by-step comparison between AI predictions and MIDI Ground Truth.
    """
    print(f"Sequential Validation: AI vs. MIDI (Window: 0-{limit_sec}s)")

    if X_features.ndim == 3:
        X_features = X_features[..., np.newaxis]

    raw_preds = model.predict(X_features, verbose=0).flatten()
    dynamic_threshold = np.clip(np.median(raw_preds), 0.3, 0.7)

    # Unify and sort MIDI Ground Truth
    midi_sequence = sorted([(k, 0) for k in gt_kick] + [(s, 1) for s in gt_snare])

    # Filter both sets based on time limit
    ai_indices = [i for i, t in enumerate(times) if t < limit_sec]
    midi_sequence_limited = [m for m in midi_sequence if m[0] < limit_sec]

    # Validation Logic
    num_comparisons = min(len(ai_indices), len(midi_sequence_limited))

    if num_comparisons == 0:
        print("Warning: No events found in the specified time window for comparison.")
        return

    print(f"\n--- SEQUENTIAL COMPARISON (0-{limit_sec}s) ---")
    print(f"{'Pos.':<4} | {'AI Time':<10} | {'AI Class.':<12} | {'MIDI GT':<12} | {'MIDI Time':<10} | {'Result'}")
    print("-" * 85)

    match_count = 0
    for j in range(num_comparisons):
        idx_ai = ai_indices[j]
        t_ai = times[idx_ai]
        p_ai = raw_preds[idx_ai]
        t_midi, label_midi_idx = midi_sequence_limited[j]

        ai_label_idx = 1 if p_ai > dynamic_threshold else 0
        ai_label_str = "SNARE" if ai_label_idx == 1 else "KICK"
        midi_label_str = "SNARE" if label_midi_idx == 1 else "KICK"

        is_match = (ai_label_idx == label_midi_idx)
        if is_match: match_count += 1
        outcome = "MATCH" if is_match else "ERROR"

        print(f"#{j+1:<3} | {t_ai:>7.2f}s  | {ai_label_str:<12} | {midi_label_str:<12} | {t_midi:>7.2f}s  | {outcome}")

    accuracy = (match_count / num_comparisons) * 100
    print("-" * 85)
    print(f"Sequential Accuracy: {accuracy:.1f}%")


def compute_symbolic_metrics_fixed(times, model, X_features, gt_kick, gt_snare, limit_sec=30.0):
    """
    Computes classification report and confusion matrix with safety checks.
    """
    if X_features.ndim == 3:
        X_features = X_features[..., np.newaxis]

    raw_preds = model.predict(X_features, verbose=0).flatten()
    dynamic_threshold = np.clip(np.median(raw_preds), 0.3, 0.7)

    # Build AI label sequence
    ai_sequence = [1 if raw_preds[i] > dynamic_threshold else 0 for i, t in enumerate(times) if t < limit_sec]

    # Build MIDI label sequence
    midi_sequence = [m[1] for m in sorted([(k, 0) for k in gt_kick] + [(s, 1) for s in gt_snare]) if m[0] < limit_sec]

    # Verify data presence to avoid ValueError
    if len(ai_sequence) == 0 or len(midi_sequence) == 0:
        print("Error: Empty sequence detected. Metrics cannot be calculated.")
        return None

    # Alignment and Metric Calculation
    num_comp = min(len(ai_sequence), len(midi_sequence))
    ai_final = np.array(ai_sequence[:num_comp])
    midi_final = np.array(midi_sequence[:num_comp])

    print(f"\nSCIENTIFIC VALIDATION REPORT (0-{limit_sec}s)")
    print("-" * 55)
    print(f"Total Sequential Events: {num_comp}")

    # Classification Metrics (Precision, Recall, F1)
    report = classification_report(midi_final, ai_final, target_names=['KICK', 'SNARE'], zero_division=0)
    print(report)

    return confusion_matrix(midi_final, ai_final)


if SONG_NAME != 'Mattia':
    if 'gt_kicks_all' in locals() and len(gt_kicks_all) > 0:
        validate_sequential_alignment(
            onset_times[:len(X_inference_cnn)],
            final_model,
            X_inference_cnn,
            gt_kicks_all,
            gt_snares_all,
            limit_sec=60.0
        )

        cm = compute_symbolic_metrics_fixed(
            onset_times[:len(X_inference_cnn)],
            final_model,
            X_inference_cnn,
            gt_kicks_all,
            gt_snares_all,
            limit_sec=60.0
        )
    else:
        print("Error: Ground Truth variables are missing or empty. Please check the MIDI loading module.")

In [ ]:
# time comparison

def validate_time_based_alignment(ai_times, model, X_features, gt_kick, gt_snare, tolerance=0.05, limit_sec=30.0):
    """
    Compares AI predictions with MIDI Ground Truth based on temporal proximity.
    Uses a greedy matching algorithm within a specified tolerance window.
    """
    print(f"\nSCIENTIFIC TIME-BASED VALIDATION (Tolerance: ±{tolerance*1000:.0f}ms)")
    print(f"Window: 0-{limit_sec}s")

    # AI Predictions Preparation
    if X_features.ndim == 3:
        X_features = X_features[..., np.newaxis]

    raw_preds = model.predict(X_features, verbose=0).flatten()
    dynamic_threshold = np.clip(np.median(raw_preds), 0.3, 0.7)

    ai_events = []
    for t, p in zip(ai_times, raw_preds):
        if t > limit_sec: break
        label = 1 if p > dynamic_threshold else 0
        ai_events.append({'time': t, 'label': label, 'prob': p})

    # MIDI Ground Truth Preparation
    midi_events = []
    for t in gt_kick:
        if t < limit_sec: midi_events.append({'time': t, 'label': 0})
    for t in gt_snare:
        if t < limit_sec: midi_events.append({'time': t, 'label': 1})
    midi_events.sort(key=lambda x: x['time'])

    print(f"AI Events detected: {len(ai_events)}")
    print(f"MIDI Events found: {len(midi_events)}")
    print("-" * 95)
    print(f"{'Pos.':<4} | {'AI Time':<10} | {'AI Class':<12} | {'MIDI Time':<10} | {'MIDI Class':<12} | {'Outcome'}")
    print("-" * 95)

    # Greedy Matching Algorithm
    true_labels = []
    pred_labels = []
    matched_midi_indices = set()
    matches = 0
    errors = 0

    for i, ai_ev in enumerate(ai_events):
        t_ai = ai_ev['time']
        label_ai = ai_ev['label']
        ai_str = "SNARE" if label_ai == 1 else "KICK"

        best_match = None
        min_dist = tolerance
        best_idx = -1

        for idx, midi_ev in enumerate(midi_events):
            if idx in matched_midi_indices: continue
            dist = abs(t_ai - midi_ev['time'])
            if dist <= min_dist:
                min_dist = dist
                best_match = midi_ev
                best_idx = idx
            if midi_ev['time'] > t_ai + tolerance:
                break

        if best_match:
            matched_midi_indices.add(best_idx)
            t_midi = best_match['time']
            label_midi = best_match['label']
            midi_str = "SNARE" if label_midi == 1 else "KICK"

            if label_ai == label_midi:
                outcome = "MATCH"
                matches += 1
            else:
                outcome = "CLASS ERR"
                errors += 1

            true_labels.append(label_midi)
            pred_labels.append(label_ai)
            print(f"#{i+1:<3} | {t_ai:>7.3f}s  | {ai_str:<12} | {t_midi:>7.3f}s  | {midi_str:<12} | {outcome}")
        else:
            print(f"#{i+1:<3} | {t_ai:>7.3f}s  | {ai_str:<12} | {'---':<10} | {'---':<12} | GHOST")

    #  Summary Statistics
    print("-" * 95)
    total_matched = matches + errors
    if total_matched > 0:
        acc = (matches / total_matched) * 100
        print(f"Classification Accuracy (on aligned events): {acc:.2f}%")
        print(f"Ghost Notes (Not in MIDI): {len(ai_events) - total_matched}")
        print(f"Missed Beats (Not detected): {len(midi_events) - total_matched}")

    if true_labels:
        print("\nDETAILED CLASSIFICATION REPORT")
        print(classification_report(true_labels, pred_labels, target_names=['KICK', 'SNARE'], zero_division=0))
        return confusion_matrix(true_labels, pred_labels)
    return None

if SONG_NAME == 'Mattia':
    if 'gt_kicks_all' in locals() and len(gt_kicks_all) > 0:
        X_val = X_inference_cnn

        # Ensure we have the confusion matrix (cm) for the next plot
        cm = validate_time_based_alignment(
            onset_times[:len(X_val)],
            final_model,
            X_val,
            gt_kicks_all,
            gt_snares_all,
            tolerance=0.10,
            limit_sec=200.0
        )
    else:
        print("Error: Ground Truth variables missing for Mattia.")

## Module 10: Final Validation Visualizations
This module provides a graphical interpretation of the scientific metrics. It includes a dual **Confusion Matrix** (absolute counts and normalized percentages) to pinpoint classification biases between Kicks and Snares. Additionally, it features a **Prediction Probability Distribution** histogram; this visualization is critical for justifying the dynamic thresholding strategy, as it shows how confidently the CNN separates the two percussive classes in the target audio track.

In [ ]:
def plot_validation_results(confusion_matrix, model, X_features):
    """
    Generates professional plots for MIR metric evaluation.
    """
    plt.figure(figsize=(18, 6))

    # Normalized Confusion Matrix
    plt.subplot(1, 2, 1)
    cm_normalized = confusion_matrix.astype('float') / confusion_matrix.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=['KICK', 'SNARE'], yticklabels=['KICK', 'SNARE'])
    plt.title("Normalized Confusion Matrix (Recall per Class)")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")

    # Prediction Probability Distribution
    plt.subplot(1, 2, 2)
    raw_preds = model.predict(X_features, verbose=0).flatten()
    dynamic_threshold = np.clip(np.median(raw_preds), 0.3, 0.7)

    plt.hist(raw_preds, bins=30, color='royalblue', edgecolor='black', alpha=0.7)
    plt.axvline(dynamic_threshold, color='red', linestyle='--', label=f'Dynamic Threshold ({dynamic_threshold:.2f})')
    plt.title("CNN Prediction Probability Distribution")
    plt.xlabel("Probability (Snare-ness)")
    plt.ylabel("Number of Onsets")
    plt.legend()

    plt.tight_layout()
    plt.show()


# Uses the confusion matrix 'cm' generated
if 'cm' in locals():
    plot_validation_results(cm, final_model, X_inference_cnn)
else:
    print("Error: Confusion matrix 'cm' not found.")